# Section 3 — Quantization (Qwen2.5-1.5B-Instruct)

Runs on Colab's free T4 GPU. Runtime menu -> Change runtime type -> T4 GPU (make sure this is set before running).

Steps:
1. Install deps
2. Run fp16 baseline
3. Run bitsandbytes 4-bit
4. Compare + view results
5. Download results_*.json to include in your submission repo


In [ ]:
!pip install -q transformers>=4.44.0 accelerate>=0.33.0 bitsandbytes>=0.43.0 tabulate

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
# If this prints False, go to Runtime -> Change runtime type -> select T4 GPU, then rerun.

In [ ]:
PROMPTS = [
    "Explain the difference between supervised and unsupervised learning in two sentences.",
    "Write a Python function that returns the nth Fibonacci number using memoization.",
    "Summarize why retrieval-augmented generation helps reduce hallucination.",
    "A customer says their food delivery order never arrived. Write a short, empathetic response as a support agent.",
    "What are the trade-offs between using FAISS and Qdrant as a vector store?",
]
MAX_NEW_TOKENS = 200
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

## 1. fp16 baseline

In [ ]:
import json, time
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.cuda.reset_peak_memory_stats()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model_fp16 = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map="cuda")
print("Loaded fp16. Memory after load (MB):", torch.cuda.max_memory_allocated() / 1024**2)

In [ ]:
def run_prompts(model, tokenizer, device="cuda"):
    results = []
    total_tokens, total_time = 0, 0.0
    for i, prompt in enumerate(PROMPTS):
        messages = [{"role": "user", "content": prompt}]
        inputs = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
        ).to(device)

        start = time.perf_counter()
        with torch.no_grad():
            output_ids = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
        elapsed = time.perf_counter() - start

        generated_ids = output_ids[0][inputs["input_ids"].shape[-1]:]
        text = tokenizer.decode(generated_ids, skip_special_tokens=True)
        n_tokens = len(generated_ids)
        tps = n_tokens / elapsed if elapsed > 0 else 0
        total_tokens += n_tokens
        total_time += elapsed

        print(f"[{i+1}/{len(PROMPTS)}] {elapsed:.2f}s | {tps:.1f} tok/s")
        print("Prompt:", prompt)
        print("Output:", text[:200], "...\n")

        results.append({"prompt": prompt, "output": text, "elapsed_sec": elapsed,
                         "n_tokens": n_tokens, "tokens_per_sec": tps})
    avg_tps = total_tokens / total_time if total_time > 0 else 0
    return results, avg_tps

In [ ]:
torch.cuda.reset_peak_memory_stats()
results_fp16, avg_tps_fp16 = run_prompts(model_fp16, tokenizer)
peak_mem_fp16 = torch.cuda.max_memory_allocated() / 1024**2

summary_fp16 = {"config": "fp16", "model_id": MODEL_ID, "peak_memory_mb": peak_mem_fp16,
                 "avg_tokens_per_sec": avg_tps_fp16, "results": results_fp16}
with open("results_fp16.json", "w") as f:
    json.dump(summary_fp16, f, indent=2)

print(f"\nPeak memory: {peak_mem_fp16:.1f} MB | Avg throughput: {avg_tps_fp16:.1f} tok/s")

In [ ]:
# Free the fp16 model before loading the 4-bit version, to keep memory measurements clean
del model_fp16
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

## 2. bitsandbytes 4-bit (NF4)

In [ ]:
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model_4bit = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map="cuda")
print("Loaded 4-bit. Memory after load (MB):", torch.cuda.max_memory_allocated() / 1024**2)

In [ ]:
torch.cuda.reset_peak_memory_stats()
results_4bit, avg_tps_4bit = run_prompts(model_4bit, tokenizer)
peak_mem_4bit = torch.cuda.max_memory_allocated() / 1024**2

summary_4bit = {"config": "bitsandbytes-4bit-nf4", "model_id": MODEL_ID, "peak_memory_mb": peak_mem_4bit,
                 "avg_tokens_per_sec": avg_tps_4bit, "results": results_4bit}
with open("results_bnb4bit.json", "w") as f:
    json.dump(summary_4bit, f, indent=2)

print(f"\nPeak memory: {peak_mem_4bit:.1f} MB | Avg throughput: {avg_tps_4bit:.1f} tok/s")

## 3. Comparison table

In [ ]:
from tabulate import tabulate

rows = [
    ["fp16 (baseline)", f"{peak_mem_fp16:.0f} MB", f"{avg_tps_fp16:.1f} tok/s"],
    ["bitsandbytes 4-bit (NF4)", f"{peak_mem_4bit:.0f} MB", f"{avg_tps_4bit:.1f} tok/s"],
]
print(tabulate(rows, headers=["Config", "Peak Memory", "Avg Throughput"], tablefmt="github"))

## 4. Download results

Run the cell below, then in the Colab file browser (left sidebar) download
`results_fp16.json` and `results_bnb4bit.json`, and place them in your
`section3_quantization/` folder alongside the scripts, then fill in the
README table with these numbers.

In [ ]:
from google.colab import files
files.download("results_fp16.json")
files.download("results_bnb4bit.json")